# Result Object — Summary & Plots

This notebook demonstrates the `Result` object's `summary()` method and `plot_actual_vs_counterfactual()` across all estimators and treatment patterns.

In [1]:
import numpy as np
import sys, os

# Make sure the package is importable when running from the tests folder
sys.path.insert(0, os.path.abspath(os.path.join('..', '..', '..')))

from causaltensor.cauest.DID import DIDPanelSolver
from causaltensor.cauest.SDID import SDIDPanelSolver
from causaltensor.cauest.OLSSyntheticControl import OLSSCPanelSolver
from causaltensor.cauest.MCNNM import MCNNMPanelSolver
from causaltensor.cauest.DebiasConvex import DCPanelSolver
from causaltensor.cauest.RobustSyntheticControl import RSCPanelSolver
from causaltensor.cauest.CovariancePCA import CovariancePCAPanelSolver
from causaltensor.utils.treatment_patterns import Z_block, Z_stagger, Z_iid, Z_adaptive

np.random.seed(42)

## 1. Synthetic panel — block treatment

20 units, 40 periods, 5 treated units from period 20 onwards.

In [2]:
N, T = 20, 40
TRUE_TAU = 2.0
years = list(range(1980, 1980 + T))   # human-readable x-axis labels

# Low-rank baseline
rng = np.random.default_rng(42)
U = rng.standard_normal((N, 3))
V = rng.standard_normal((T, 3))
M = U @ V.T

Z_blk = Z_block(M, m1=5, m2=20, rng=rng)
O_blk = M + TRUE_TAU * Z_blk + 0.1 * rng.standard_normal((N, T))

treated_units = list(np.where(np.any(Z_blk > 0, axis=1))[0])
control_units = list(np.where(np.all(Z_blk == 0, axis=1))[0])
print(f'Treated units: {treated_units}')
print(f'Control units: {control_units[:5]} ...')

Treated units: [0, 5, 6, 7, 14]
Control units: [1, 2, 3, 4, 8] ...


### 1a. DID — summary

In [3]:
res_did = DIDPanelSolver(O_blk, Z_blk).fit()
res_did.summary();

  DIDResult
  Panel                     : 20 units x 40 periods
  Treated cells             : 100 / 800  (12.5%)
  Treated units             : 5
  Treatment pattern         : Block  (T0 = 20)
--------------------------------------------------------
  ATT (tau)                 : 1.96807

  --- Fit diagnostics ---
  Untreated R2              : 0.04806
  Control RMSE              : 1.2856
  Pre-exposure RMSE         : 1.24137  (700 cells)
  RMSPE ratio               : 1.585

  --- Model internals ---
  row_FE (unit)           : min=-0.2602  mean=-0.01776  max=0.2758
  col_FE (time)           : min=-0.2675  mean=8.327e-18  max=0.5098


### 1b. DID — actual vs counterfactual (treated unit)

In [4]:
fig = res_did.plot_actual_vs_counterfactual(
    unit=treated_units[0],
    unit_label=f'Treated Unit {treated_units[0]}',
    time_labels=[str(y) for y in years],
    title='DID — Block Treatment'
)
fig.show()

### 1c. DID — actual vs counterfactual (control unit, for sanity check)

In [5]:
fig = res_did.plot_actual_vs_counterfactual(
    unit=control_units[0],
    unit_label=f'Control Unit {control_units[0]}',
    time_labels=[str(y) for y in years],
    title='DID — Control Unit (gap should be ~0)'
)
fig.show()

## 2. Staggered adoption

In [6]:
Z_stg = Z_stagger(M, m1=6, min_start=12, rng=rng)
O_stg = M + TRUE_TAU * Z_stg + 0.1 * rng.standard_normal((N, T))

treated_stg = list(np.where(np.any(Z_stg > 0, axis=1))[0])

res_sdid = SDIDPanelSolver(O_stg, Z_stg).fit()
res_sdid.summary();

  SDIDResult
  Panel                     : 20 units x 40 periods
  Treated cells             : 91 / 800  (11.4%)
  Treated units             : 6
  Treatment pattern         : Staggered  (T0: 16..33)
--------------------------------------------------------
  ATT (tau)                 : 1.99314

  --- Fit diagnostics ---
  Untreated R2              : -0.05695
  Control RMSE              : 1.26116
  Pre-exposure RMSE         : 1.26275  (709 cells)
  RMSPE ratio               : 1.578

  --- Model internals ---
  unit_weights            : 20 nonzero  (top: unit 0, w=0.1667)
    (full array)          : result.unit_weights
  time_weights            : 22 nonzero
    (full array)          : result.time_weights


In [7]:
# Show two treated units with different treatment start times
for u in treated_stg[:2]:
    t0 = int(np.where(Z_stg[u] > 0)[0][0])
    fig = res_sdid.plot_actual_vs_counterfactual(
        unit=u,
        unit_label=f'Unit {u}  (T0 = {years[t0]})',
        time_labels=[str(y) for y in years],
        title='SDID — Staggered Adoption'
    )
    fig.show()

## 3. OLS Synthetic Control — donor weights visible in summary

In [8]:
res_sc = OLSSCPanelSolver(O_blk, Z_blk, pval=True).fit()
res_sc.summary();

  OLSSCResult
  Panel                     : 20 units x 40 periods
  Treated cells             : 100 / 800  (12.5%)
  Treated units             : 5
  Treatment pattern         : Block  (T0 = 20)
--------------------------------------------------------
  ATT (tau)                 : 2.02946

  --- Fit diagnostics ---
  Untreated R2              : 0.9814
  Control RMSE              : 0
  Pre-exposure RMSE         : 0.173679  (700 cells)
  RMSPE ratio               : 11.69

  --- Model internals ---
  n_treated_units         : 5
  n_donor_units           : 15
    (weights per unit)    : result.beta  (list of arrays)
    (unit indices)        : result.treatment_units, result.control_units

  per-unit ATT:
    unit 0     : tau=1.987  p=0.25  (3 donors)
    unit 5     : tau=2.066  p=0.05  (3 donors)
    unit 6     : tau=2.015  p=0.2  (6 donors)
    unit 7     : tau=2.047  p=0.1  (3 donors)
    unit 14    : tau=2.032  p=0.15  (3 donors)


In [9]:
# Inspect donor weights directly
print('Donor weights for treated unit', res_sc.treatment_units[0])
weights = res_sc.beta[0]
donors  = res_sc.control_units
for d, w in sorted(zip(donors, weights), key=lambda x: -x[1]):
    if w > 1e-4:
        print(f'  Unit {d}: {w:.4f}')

Donor weights for treated unit 0
  Unit 8: 0.4336
  Unit 19: 0.2984
  Unit 17: 0.2680


In [10]:
fig = res_sc.plot_actual_vs_counterfactual(
    unit=treated_units[0],
    unit_label=f'Treated Unit {treated_units[0]}',
    time_labels=[str(y) for y in years],
    title='OLS Synthetic Control — Block Treatment'
)
fig.show()

## 4. Matrix completion (MC-NNM)

In [11]:
res_mc = MCNNMPanelSolver(O_blk, Z_blk).fit()
res_mc.summary();

  MCNNMResult
  Panel                     : 20 units x 40 periods
  Treated cells             : 100 / 800  (12.5%)
  Treated units             : 5
  Treatment pattern         : Block  (T0 = 20)
--------------------------------------------------------
  ATT (tau)                 : 1.98465

  --- Fit diagnostics ---
  Untreated R2              : 0.9915
  Control RMSE              : 0.110627
  Pre-exposure RMSE         : 0.117029  (700 cells)
  RMSPE ratio               : 16.96

  --- Model internals ---
  rank(M)                 : 4
  row_FE (unit)           : min=-0.2604  mean=-0.02009  max=0.2755
  col_FE (time)           : min=-0.3151  mean=0.0002514  max=0.6211


In [12]:
fig = res_mc.plot_actual_vs_counterfactual(
    unit=treated_units[1],
    unit_label=f'Treated Unit {treated_units[1]}',
    time_labels=[str(y) for y in years],
    title='MC-NNM — Block Treatment'
)
fig.show()

## 5. DC-PR with sandwich SE

In [13]:
res_dc = DCPanelSolver(O_blk, Z_blk).fit()
res_dc.summary();

  DCResult
  Panel                     : 20 units x 40 periods
  Treated cells             : 100 / 800  (12.5%)
  Treated units             : 5
  Treatment pattern         : Block  (T0 = 20)
--------------------------------------------------------
  ATT (tau)                 : 2.00125
  Std dev (tau)             : 0.00816088
  Inference method          : sandwich SE

  --- Fit diagnostics ---
  Untreated R2              : 0.9975
  Control RMSE              : 0.0640268
  Pre-exposure RMSE         : 0.0637661  (700 cells)
  RMSPE ratio               : 31.38

  --- Model internals ---
  rank(M)                 : 7


In [14]:
fig = res_dc.plot_actual_vs_counterfactual(
    unit=treated_units[0],
    unit_label=f'Treated Unit {treated_units[0]}',
    time_labels=[str(y) for y in years],
    title='DC-PR — std(tau) shown in annotation'
)
fig.show()

## 6. Non-monotone treatment patterns

For IID and adaptive treatment `control_rmse` is N/A (every unit has at least one treated cell).  
The note in the summary redirects to `untreated_r2` as the better diagnostic.

In [15]:
# IID
Z_i = Z_iid(M, p_treat=0.15, rng=rng)
O_i = M + TRUE_TAU * Z_i + 0.1 * rng.standard_normal((N, T))
res_iid = DCPanelSolver(O_i, Z_i).fit()
res_iid.summary();

  DCResult
  Panel                     : 20 units x 40 periods
  Treated cells             : 125 / 800  (15.6%)
  Treated units             : 20
  Treatment pattern         : Non-monotone (IID/adaptive)
--------------------------------------------------------
  ATT (tau)                 : 1.93103
  Std dev (tau)             : 0.00811771
  Inference method          : sandwich SE

  --- Fit diagnostics ---
  Untreated R2              : 0.9999
  Control RMSE              : N/A  (no pure control units)
  Pre-exposure RMSE         : 0.0100472  (63 cells)
    [non-monotone note]     : pre-period before first treatment only;
                              prefer Untreated R2 for full picture
  RMSPE ratio               : 192.2

  --- Model internals ---
  rank(M)                 : 18


In [16]:
fig = res_iid.plot_actual_vs_counterfactual(
    unit=0,
    unit_label='Unit 0',
    time_labels=[str(y) for y in years],
    title='DC-PR — IID Treatment (individual cells shaded)'
)
fig.show()

In [17]:
# Adaptive
Z_a = Z_adaptive(M, lookback_a=3, duration_b=4)
O_a = M + TRUE_TAU * Z_a + 0.1 * rng.standard_normal((N, T))
res_adp = DCPanelSolver(O_a, Z_a).fit()
res_adp.summary();

  DCResult
  Panel                     : 20 units x 40 periods
  Treated cells             : 311 / 800  (38.9%)
  Treated units             : 20
  Treatment pattern         : Non-monotone (IID/adaptive)
--------------------------------------------------------
  ATT (tau)                 : 2.00007
  Std dev (tau)             : 0.00538723
  Inference method          : sandwich SE

  --- Fit diagnostics ---
  Untreated R2              : 0.9993
  Control RMSE              : N/A  (no pure control units)
  Pre-exposure RMSE         : 0.0351073  (104 cells)
    [non-monotone note]     : pre-period before first treatment only;
                              prefer Untreated R2 for full picture
  RMSPE ratio               : 56.97

  --- Model internals ---
  rank(M)                 : 14


In [18]:
fig = res_adp.plot_actual_vs_counterfactual(
    unit=0,
    unit_label='Unit 0',
    time_labels=[str(y) for y in years],
    title='DC-PR — Adaptive Treatment (episode shading)'
)
fig.show()

## 7. Direct attribute access

All diagnostics and internals are accessible directly on the result object — no need to call `summary()`.

In [19]:
res = res_did   # any result object

print('--- Core outputs ---')
print('tau:              ', res.tau)
print('baseline shape:   ', res.baseline.shape)
print('residuals shape:  ', res.residuals.shape)
print('effect_matrix nnz:', np.sum(res.effect_matrix != 0))

print()
print('--- Treatment info ---')
print('z_pattern:        ', res.z_pattern)
print('Z shape:          ', res.Z.shape)

print()
print('--- Diagnostics ---')
print('untreated_r2:     ', res.untreated_r2)
print('untreated_rmse:   ', res.untreated_rmse)   # still accessible as property
print('control_rmse:     ', res.control_rmse)
print('pre_exposure_rmse:', res.pre_exposure_rmse)
print('rmspe_ratio:      ', res.rmspe_ratio)

--- Core outputs ---
tau:               1.9680703549475835
baseline shape:    (20, 40)
residuals shape:   (20, 40)
effect_matrix nnz: 100

--- Treatment info ---
z_pattern:         block
Z shape:           (20, 40)

--- Diagnostics ---
untreated_r2:      0.04806116349955192
untreated_rmse:    1.2413707246062258
control_rmse:      1.2855993986178744
pre_exposure_rmse: 1.2413707246062258
rmspe_ratio:       1.5854009732442125


In [20]:
# SDID weights
res2 = SDIDPanelSolver(O_blk, Z_blk).fit()
print('unit_weights:', res2.unit_weights.round(4))
print('time_weights:', res2.time_weights.round(4))

unit_weights: [0.2    0.0571 0.0686 0.0803 0.0748 0.2    0.2    0.2    0.0707 0.0684
 0.0519 0.0757 0.0699 0.0667 0.2    0.0738 0.0634 0.0582 0.0727 0.0475]
time_weights: [0.0778 0.     0.     0.0411 0.0916 0.1399 0.1703 0.     0.0187 0.
 0.2006 0.     0.0711 0.     0.1054 0.     0.0045 0.     0.     0.0789
 0.05   0.05   0.05   0.05   0.05   0.05   0.05   0.05   0.05   0.05
 0.05   0.05   0.05   0.05   0.05   0.05   0.05   0.05   0.05   0.05  ]


In [21]:
# OLS SC donor weights
print('treatment_units:', res_sc.treatment_units)
print('control_units:  ', res_sc.control_units)
print('beta[0]:        ', np.round(res_sc.beta[0], 4))

treatment_units: [0, 5, 6, 7, 14]
control_units:   [1, 2, 3, 4, 8, 9, 10, 11, 12, 13, 15, 16, 17, 18, 19]
beta[0]:         [0.     0.     0.     0.     0.4336 0.     0.     0.     0.     0.
 0.     0.     0.268  0.     0.2984]
